# Putting It All Together: A Data Cleaning Case Study

## The Scenario

In this notebook, we'll work through cleaning a messy dataset from start to finish. The data simulates what you might receive from a data vendor - it has the kinds of issues you'll encounter in practice:

- Missing values
- Duplicates  
- Outliers (some clearly data errors)
- Inconsistent text formatting
- Invalid values (negative volumes, infinities)

We'll follow a systematic process:
1. **Diagnose** - Understand what's wrong
2. **Plan** - Decide how to fix each issue
3. **Clean** - Execute the fixes in the right order
4. **Validate** - Confirm the data is now usable
5. **Document** - Record what we did

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# We'll set seeds to make our "messy" data reproducible
np.random.seed(42)

print("Environment ready. Let's see what we're dealing with...")

## Step 0: Load the Data

We'll generate synthetic data with intentional problems. In practice, you'd load this from a CSV or database.

In [ ]:
def generate_messy_stock_data():
    """Generate realistic messy stock data with various quality issues."""
    np.random.seed(42)
    
    # Base parameters for each stock
    stock_params = {
        'AAPL': {'start': 150, 'vol': 0.25, 'drift': 0.12},
        'MSFT': {'start': 350, 'vol': 0.22, 'drift': 0.10},
        'GOOGL': {'start': 140, 'vol': 0.28, 'drift': 0.08},
        'AMZN': {'start': 170, 'vol': 0.30, 'drift': 0.15},
        'NVDA': {'start': 450, 'vol': 0.45, 'drift': 0.25},
    }
    
    dates = pd.date_range('2023-01-01', '2024-06-30', freq='B')  # ~375 trading days
    all_data = []
    
    for ticker, params in stock_params.items():
        # Generate clean price series using GBM
        daily_returns = np.random.normal(
            params['drift'] / 252,
            params['vol'] / np.sqrt(252),
            len(dates)
        )
        prices = params['start'] * np.exp(np.cumsum(daily_returns))
        
        # Generate OHLCV
        intraday_vol = params['vol'] / np.sqrt(252) * 0.3
        df = pd.DataFrame({
            'date': dates,
            'ticker': ticker,
            'open': prices * (1 + np.random.normal(0, intraday_vol, len(dates))),
            'high': prices * (1 + np.abs(np.random.normal(0, intraday_vol * 2, len(dates)))),
            'low': prices * (1 - np.abs(np.random.normal(0, intraday_vol * 2, len(dates)))),
            'close': prices,
            'volume': np.random.randint(10_000_000, 100_000_000, len(dates)),
            'sector': {'AAPL': 'Technology', 'MSFT': 'Technology', 'GOOGL': 'Technology',
                      'AMZN': 'Consumer', 'NVDA': 'Technology'}[ticker],
            'source': np.random.choice(['Bloomberg', 'Reuters', 'Vendor_A'], len(dates))
        })
        
        # Ensure OHLC relationships are valid
        df['high'] = df[['open', 'high', 'close']].max(axis=1)
        df['low'] = df[['open', 'low', 'close']].min(axis=1)
        
        all_data.append(df)
    
    df = pd.concat(all_data, ignore_index=True)
    
    # ============================================
    # NOW LET'S MAKE IT MESSY!
    # ============================================
    
    # 1. MISSING VALUES (~5% of closing prices)
    missing_mask = np.random.random(len(df)) < 0.05
    df.loc[missing_mask, 'close'] = np.nan
    
    # 2. OUTLIERS (some prices multiplied by 100 - decimal point error)
    outlier_mask = np.random.random(len(df)) < 0.01
    df.loc[outlier_mask, 'close'] = df.loc[outlier_mask, 'close'] * 100
    
    # 3. DUPLICATES (about 3% duplicate rows)
    dup_indices = np.random.choice(len(df), int(len(df) * 0.03), replace=False)
    duplicates = df.iloc[dup_indices].copy()
    df = pd.concat([df, duplicates], ignore_index=True)
    
    # 4. TICKER NAME ISSUES (some have HTML tags, inconsistent case)
    messy_ticker_mask = np.random.random(len(df)) < 0.08
    messy_tickers = df.loc[messy_ticker_mask, 'ticker'].copy()
    messy_options = [
        lambda x: f'<b>{x}</b>',
        lambda x: x.lower(),
        lambda x: f' {x} ',
        lambda x: f'{x.lower()} '
    ]
    df.loc[messy_ticker_mask, 'ticker'] = messy_tickers.apply(
        lambda x: np.random.choice(messy_options)(x)
    )
    
    # 5. INCONSISTENT SECTOR NAMES
    sector_variations = {
        'Technology': ['Technology', 'Tech', 'TECH', 'technology', 'Information Technology'],
        'Consumer': ['Consumer', 'Consumer Discretionary', 'consumer', 'CONSUMER', 'Retail']
    }
    def randomize_sector(sector):
        if sector in sector_variations:
            return np.random.choice(sector_variations[sector])
        return sector
    
    df['sector'] = df['sector'].apply(randomize_sector)
    
    # 6. NEGATIVE VOLUMES (clearly impossible)
    neg_vol_mask = np.random.random(len(df)) < 0.02
    df.loc[neg_vol_mask, 'volume'] = -df.loc[neg_vol_mask, 'volume']
    
    # 7. INF VALUES in some volumes
    # Floating-point storage is required to represent infinity.
    df['volume'] = df['volume'].astype(float)
    inf_mask = np.random.random(len(df)) < 0.005
    df.loc[inf_mask, 'volume'] = np.inf
    
    # 8. SOME NULL SECTORS
    null_sector_mask = np.random.random(len(df)) < 0.03
    df.loc[null_sector_mask, 'sector'] = None
    
    # Shuffle to mix everything up
    df = df.sample(frac=1).reset_index(drop=True)
    
    return df

# Generate the messy data
raw_data = generate_messy_stock_data()
print(f"Received {len(raw_data)} rows of data")
print(f"Date range: {raw_data['date'].min()} to {raw_data['date'].max()}")

In [ ]:
# First look at the data
raw_data.head(20)

---

# Step 1: Initial Diagnosis

Before fixing anything, we need to understand what's wrong. Let's use the tools from the previous notebook.

In [ ]:
# Basic info
print("=" * 60)
print("DATA DIAGNOSIS REPORT")
print("=" * 60)
print(f"\nTotal rows: {len(raw_data)}")
print(f"Total columns: {len(raw_data.columns)}")
print(f"\nColumn types:")
raw_data.info()

In [ ]:
# Summary statistics - look for suspicious values
print("\n" + "=" * 60)
print("NUMERICAL SUMMARY")
print("=" * 60)
raw_data.describe()

In [ ]:
# Check categorical columns
print("\n" + "=" * 60)
print("CATEGORICAL ANALYSIS")
print("=" * 60)

print(f"\nUnique tickers ({raw_data['ticker'].nunique()}):")
print(raw_data['ticker'].value_counts())

print(f"\nUnique sectors ({raw_data['sector'].nunique()}):")
print(raw_data['sector'].value_counts())

## Initial Observations

Let's document what we found (this is crucial - write it down!):

**Red Flags Detected:**
1. **Too many rows**: We have more rows than expected (5 stocks × ~375 days = ~1875, but we have more)
2. **Ticker variations**: Way more than 5 unique tickers - there are formatting issues
3. **Close price max**: Maximum close is way too high (>10,000) - likely decimal errors
4. **Volume**: Minimum is negative (!!) and max is inf
5. **Sectors**: Multiple variations of the same sector name
6. **Missing values**: Non-null counts show gaps in close and sector

## Visual Diagnosis

Numbers tell part of the story. Let's see the problems.

In [ ]:
# First, let's try to plot raw close prices
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Raw close prices over time (all tickers mixed)
ax = axes[0, 0]
raw_data_sorted = raw_data.sort_values('date')
ax.scatter(raw_data_sorted['date'], raw_data_sorted['close'], alpha=0.3, s=1)
ax.set_title('Raw Close Prices - ALL DATA')
ax.set_ylabel('Price')
ax.set_xlabel('Date')

# Histogram of close prices - shows outliers
ax = axes[0, 1]
raw_data['close'].dropna().hist(bins=100, ax=ax, edgecolor='black')
ax.set_title('Distribution of Close Prices')
ax.set_xlabel('Price')
ax.set_ylabel('Frequency')

# Volume distribution
ax = axes[1, 0]
# Filter out inf for plotting
vol_clean = raw_data['volume'].replace([np.inf, -np.inf], np.nan).dropna()
ax.hist(vol_clean, bins=100, edgecolor='black')
ax.set_title('Distribution of Volume (excl. inf)')
ax.set_xlabel('Volume')
ax.set_ylabel('Frequency')

# Box plot by ticker (raw tickers - will be messy)
ax = axes[1, 1]
# Just show first 10 unique tickers for readability
top_tickers = raw_data['ticker'].value_counts().head(10).index
raw_data[raw_data['ticker'].isin(top_tickers)].boxplot(column='close', by='ticker', ax=ax)
ax.set_title('Close Prices by Ticker (top 10)')
plt.suptitle('')  # Remove automatic title

plt.tight_layout()

**Visual Insights:**
- The scatter plot shows massive outliers (prices > 10,000)
- The histogram is completely skewed by outliers
- Volume has negative values visible in the left tail
- Box plots show ticker variations causing multiple categories

---

# Step 2: Create a Cleaning Plan

Based on our diagnosis, here's our cleaning plan:

| Issue | Detection | Solution |
|-------|-----------|----------|
| 1. Ticker formatting | `value_counts()` shows variations | Strip, uppercase, remove HTML |
| 2. Duplicates | Row count > expected | `drop_duplicates()` |
| 3. Price outliers | Max >> expected | Cap at 3 std or use close/open ratio |
| 4. Missing close | `.isnull()` | Forward fill (time series) |
| 5. Invalid volume | < 0 or inf | Replace with NaN, then median |
| 6. Sector variations | `value_counts()` | Standardize mapping |
| 7. Missing sector | `.isnull()` | Fill with 'Unknown' or infer from ticker |

**Order matters!** We should:
1. Fix text issues first (so we can group properly)
2. Remove duplicates
3. Then handle missing values and outliers

---

# Step 3: Execute the Cleaning

Let's clean systematically, verifying each step.

In [ ]:
# Always work on a copy!
df = raw_data.copy()

# Track our changes
cleaning_log = []
initial_rows = len(df)

print(f"Starting with {initial_rows} rows")

### 3.1 Clean Ticker Names

In [ ]:
# Before
print("BEFORE - Unique tickers:")
print(df['ticker'].value_counts())

In [ ]:
# Clean tickers: strip whitespace, uppercase, remove HTML
df['ticker'] = (df['ticker']
                .str.strip()
                .str.upper()
                .str.replace(r'<[^>]+>', '', regex=True))

# Verify
print("AFTER - Unique tickers:")
print(df['ticker'].value_counts())

cleaning_log.append(f"Standardized ticker names: now have {df['ticker'].nunique()} unique tickers")

### 3.2 Clean Sector Names

In [ ]:
# Before
print("BEFORE - Unique sectors:")
print(df['sector'].value_counts(dropna=False))

In [ ]:
# Create a mapping for sector standardization
sector_mapping = {
    'Technology': 'Technology',
    'Tech': 'Technology',
    'TECH': 'Technology',
    'technology': 'Technology',
    'Information Technology': 'Technology',
    'Consumer': 'Consumer',
    'Consumer Discretionary': 'Consumer',
    'consumer': 'Consumer',
    'CONSUMER': 'Consumer',
    'Retail': 'Consumer'
}

# Apply mapping
df['sector'] = df['sector'].replace(sector_mapping)

# Fill remaining nulls based on ticker
ticker_sector_map = {
    'AAPL': 'Technology',
    'MSFT': 'Technology',
    'GOOGL': 'Technology',
    'NVDA': 'Technology',
    'AMZN': 'Consumer'
}

# Fill missing sectors from ticker map
null_sector_mask = df['sector'].isnull()
df.loc[null_sector_mask, 'sector'] = df.loc[null_sector_mask, 'ticker'].map(ticker_sector_map)

# Verify
print("AFTER - Unique sectors:")
print(df['sector'].value_counts(dropna=False))

cleaning_log.append(f"Standardized sectors: now have {df['sector'].nunique()} unique sectors")

### 3.3 Remove Duplicates

In [ ]:
# Check for duplicates
dups_count = df.duplicated(subset=['date', 'ticker']).sum()
print(f"Found {dups_count} duplicate date-ticker combinations")

# Show some examples
dup_mask = df.duplicated(subset=['date', 'ticker'], keep=False)
print("\nExample duplicates:")
df[dup_mask].sort_values(['ticker', 'date']).head(10)

In [ ]:
# Remove duplicates - keep first occurrence
rows_before = len(df)
df = df.drop_duplicates(subset=['date', 'ticker'], keep='first')
rows_after = len(df)

print(f"Removed {rows_before - rows_after} duplicate rows")
print(f"Remaining rows: {rows_after}")

cleaning_log.append(f"Removed {rows_before - rows_after} duplicates")

### 3.4 Handle Price Outliers

In [ ]:
# Let's look at the extreme prices
print("Top 10 highest close prices:")
print(df.nlargest(10, 'close')[['date', 'ticker', 'open', 'high', 'low', 'close']])

In [ ]:
# Our heuristic: if close > high, it's definitely wrong (close can't exceed day's high)
# Also: if close / open > 2 (100% gain in a day), suspicious for these stocks

impossible_mask = df['close'] > df['high']
print(f"Impossible prices (close > high): {impossible_mask.sum()}")

suspicious_mask = (df['close'] / df['open']) > 2
print(f"Suspicious prices (>100% daily gain): {suspicious_mask.sum()}")

In [ ]:
# Strategy: For impossible/suspicious prices, replace with average of high and low
# (This is a reasonable estimate when the close is clearly wrong)

outlier_mask = impossible_mask | suspicious_mask
outlier_count = outlier_mask.sum()

# Replace outliers with (high + low) / 2
df.loc[outlier_mask, 'close'] = (df.loc[outlier_mask, 'high'] + df.loc[outlier_mask, 'low']) / 2

print(f"Corrected {outlier_count} outlier prices")

cleaning_log.append(f"Corrected {outlier_count} outlier close prices using (high+low)/2")

### 3.5 Handle Missing Close Prices

In [ ]:
# Check missing close prices
missing_close = df['close'].isnull().sum()
print(f"Missing close prices: {missing_close}")

# See the pattern
print("\nMissing close by ticker:")
print(df[df['close'].isnull()].groupby('ticker').size())

In [ ]:
# Strategy: Forward fill within each ticker (use last known price)
# First sort by ticker and date
df = df.sort_values(['ticker', 'date'])

# Forward fill within groups
df['close'] = df.groupby('ticker')['close'].ffill()

# Check if any still missing (first days for each stock might still be null)
still_missing = df['close'].isnull().sum()
print(f"Still missing after ffill: {still_missing}")

# Backward fill any remaining
if still_missing > 0:
    df['close'] = df.groupby('ticker')['close'].bfill()

print(f"Final missing: {df['close'].isnull().sum()}")

cleaning_log.append(f"Filled {missing_close} missing close prices using forward/backward fill")

### 3.6 Handle Invalid Volumes

In [ ]:
# Check volume issues
negative_vol = (df['volume'] < 0).sum()
inf_vol = np.isinf(df['volume']).sum()

print(f"Negative volumes: {negative_vol}")
print(f"Infinite volumes: {inf_vol}")

In [ ]:
# Strategy: Replace invalid volumes with median volume per ticker

# First, mark invalid as NaN
invalid_vol_mask = (df['volume'] < 0) | np.isinf(df['volume'])
df.loc[invalid_vol_mask, 'volume'] = np.nan

# Calculate median per ticker
median_volumes = df.groupby('ticker')['volume'].transform('median')

# Fill NaN volumes with ticker median
df['volume'] = df['volume'].fillna(median_volumes)

print(f"Corrected {invalid_vol_mask.sum()} invalid volumes")

cleaning_log.append(f"Corrected {invalid_vol_mask.sum()} invalid volumes using ticker median")

---

# Step 4: Validate the Cleaned Data

Let's verify our cleaning worked and the data is now usable.

In [ ]:
print("=" * 60)
print("CLEANING LOG")
print("=" * 60)
for i, entry in enumerate(cleaning_log, 1):
    print(f"{i}. {entry}")

print(f"\nRows: {initial_rows} -> {len(df)} ({initial_rows - len(df)} removed)")

In [ ]:
# Final data quality check
print("\n" + "=" * 60)
print("FINAL DATA QUALITY CHECK")
print("=" * 60)

df.info()

In [ ]:
# Summary stats - should look reasonable now
print("\nSummary Statistics:")
df.describe()

In [ ]:
# Categorical check
print("\nTicker distribution:")
print(df['ticker'].value_counts())

print("\nSector distribution:")
print(df['sector'].value_counts())

In [ ]:
# Visual validation - BEFORE vs AFTER
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price distribution - BEFORE (from raw_data)
ax = axes[0, 0]
raw_data['close'].dropna().hist(bins=100, ax=ax, edgecolor='black', alpha=0.7)
ax.set_title('BEFORE: Close Price Distribution')
ax.set_xlabel('Price')

# Price distribution - AFTER
ax = axes[0, 1]
df['close'].hist(bins=100, ax=ax, edgecolor='black', alpha=0.7, color='green')
ax.set_title('AFTER: Close Price Distribution')
ax.set_xlabel('Price')

# Time series - BEFORE
ax = axes[1, 0]
for ticker in raw_data['ticker'].str.strip().str.upper().str.replace(r'<[^>]+>', '', regex=True).unique()[:5]:
    mask = raw_data['ticker'].str.strip().str.upper().str.replace(r'<[^>]+>', '', regex=True) == ticker
    temp = raw_data[mask].sort_values('date')
    ax.plot(temp['date'], temp['close'], label=ticker, alpha=0.7)
ax.set_title('BEFORE: Stock Prices Over Time')
ax.set_ylabel('Price')
ax.legend()

# Time series - AFTER
ax = axes[1, 1]
for ticker in df['ticker'].unique():
    temp = df[df['ticker'] == ticker].sort_values('date')
    ax.plot(temp['date'], temp['close'], label=ticker, alpha=0.7)
ax.set_title('AFTER: Stock Prices Over Time')
ax.set_ylabel('Price')
ax.legend()

plt.tight_layout()
plt.suptitle('Data Cleaning: Before vs After', y=1.02, fontsize=14, fontweight='bold')

## Verification

Now we can use this data for analysis. Let's do a quick sanity check:

In [ ]:
# Calculate returns
df = df.sort_values(['ticker', 'date'])
df['returns'] = df.groupby('ticker')['close'].pct_change()

# Summary by ticker
summary = df.groupby('ticker').agg({
    'returns': ['mean', 'std'],
    'volume': 'mean',
    'close': ['first', 'last']
}).round(4)

summary.columns = ['Avg Daily Return', 'Volatility', 'Avg Volume', 'First Price', 'Last Price']
summary['Total Return'] = ((summary['Last Price'] / summary['First Price']) - 1).round(4)
summary['Annualized Vol'] = (summary['Volatility'] * np.sqrt(252)).round(4)

print("Stock Summary (Clean Data):")
summary

In [ ]:
# Correlation matrix of returns
returns_pivot = df.pivot_table(index='date', columns='ticker', values='returns')

fig, ax = plt.subplots(figsize=(10, 8))
corr = returns_pivot.corr()
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns)
ax.set_yticklabels(corr.columns)

# Add correlation values
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center')

plt.colorbar(im)
ax.set_title('Return Correlations (Clean Data)')
plt.tight_layout()

---

# Step 5: Save and Document

Always save your clean data and document what you did!

In [ ]:
# Save clean data
df.to_csv('clean_stock_data.csv', index=False)
print("Clean data saved to 'clean_stock_data.csv'")

# Save cleaning log
with open('cleaning_log.txt', 'w') as f:
    f.write("DATA CLEANING LOG\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Original rows: {initial_rows}\n")
    f.write(f"Final rows: {len(df)}\n")
    f.write("\nCleaning Steps:\n")
    for i, entry in enumerate(cleaning_log, 1):
        f.write(f"{i}. {entry}\n")

print("Cleaning log saved to 'cleaning_log.txt'")

---

# Summary: The Data Cleaning Workflow

Here's the workflow we followed:

```
1. DIAGNOSE
   ├── df.info() - types and nulls
   ├── df.describe() - suspicious stats
   ├── value_counts() - categorical issues
   └── Visualize - see the problems

2. PLAN
   ├── Document all issues found
   ├── Decide on strategy for each
   └── Determine order of operations

3. CLEAN (in order!)
   ├── Fix text/categorical first
   ├── Remove duplicates
   ├── Handle outliers
   └── Fill missing values

4. VALIDATE
   ├── Re-run diagnostics
   ├── Compare before/after
   └── Sanity check with analysis

5. DOCUMENT & SAVE
   ├── Save clean data
   ├── Log all steps taken
   └── Make reproducible!
```

## Key Takeaways

1. **Visualize early** - Plots often reveal issues that summary stats miss
2. **Work on a copy** - Keep the raw data intact
3. **Order matters** - Clean text before deduplication, remove duplicates before filling missing values
4. **Use domain knowledge** - We knew stock close prices can't exceed the daily high
5. **Document your steps** - You (or someone else) will need to understand what you did later
6. **Validate at the end** - Before/after comparison confirms the cleaning worked

---

# Mini-Challenge (Optional)

Want more practice? Try these exercises:

1. **Add a new cleaning step**: The `source` column has different data vendors. Standardize them to just two categories: 'Bloomberg' and 'Other'.

2. **Improve outlier detection**: Instead of using close/open ratio, implement Z-score detection within each ticker.

3. **Add data validation**: Write a function that checks if OHLC relationships are valid (Open/Close should be between Low and High).

4. **Calculate more metrics**: Add columns for daily range, rolling volatility, and 20-day moving average.

Happy cleaning!